# Interactive 2D MST embedding

Explore how the 2D MNIST embedding changes as you adjust the sample count and estimator settings. Load a fixed, stratified pool once, then move the sliders and click **Fit embedding** to run a new fit. Fitting happens on button click so dragging controls does not start repeated runs. Larger samples, more MSTs, and more epochs take longer.

Install notebook dependencies with `python -m pip install -e ".[notebook]"`. The first data cell downloads MNIST from OpenML if it is not cached.


In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import clear_output, display, HTML
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import sys
import time
import traceback

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "digits_sweep.py").is_file()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root / "notebooks"))

from digits_sweep import load_mnist_data
from mst_embedding import IteratedMSTEmbedding


## Load a reusable stratified sample pool

The sample-count slider selects a balanced prefix from this fixed pool, so adjusting it does not download or preprocess MNIST again.


In [ ]:
MAX_INSTANCES = 4000
X_pool, labels_pool = load_mnist_data(
    sample_size=MAX_INSTANCES,
    random_state=42,
)

# Interleave shuffled examples from each digit class so every slider value
# (a multiple of 10) has balanced class counts.
rng = np.random.RandomState(42)
class_indices = [
    rng.permutation(np.flatnonzero(labels_pool == label))
    for label in np.unique(labels_pool)
]
per_class = min(len(indices) for indices in class_indices)
balanced_order = np.stack(
    [indices[:per_class] for indices in class_indices], axis=1
).reshape(-1)
X_pool = X_pool[balanced_order]
labels_pool = labels_pool[balanced_order]
MAX_INSTANCES = len(X_pool)
print(f"Loaded {MAX_INSTANCES:,} balanced, standardized samples.")


## Adjust the embedding

The controls change the dataset size, number of MSTs, rank-weight exponent, optimization epochs, learning rate, repulsion strength, and negative sampling ratio. The sample-count slider ranges from 500 to the loaded pool size.


In [ ]:
sample_count = widgets.IntSlider(
    value=2000, min=500, max=MAX_INSTANCES, step=250,
    description="Instances", continuous_update=False,
    style={"description_width": "initial"},
)
n_msts = widgets.IntSlider(
    value=8, min=1, max=20, step=1,
    description="MSTs", continuous_update=False,
    style={"description_width": "initial"},
)
rank_exponent = widgets.FloatSlider(
    value=1.0, min=0.0, max=3.0, step=0.25,
    description="Rank exponent", continuous_update=False,
    style={"description_width": "initial"},
)
n_epochs = widgets.IntSlider(
    value=500, min=100, max=1000, step=100,
    description="Epochs", continuous_update=False,
    style={"description_width": "initial"},
)
learning_rate = widgets.FloatSlider(
    value=0.05, min=0.01, max=0.10, step=0.01,
    readout_format=".2f", description="Learning rate",
    continuous_update=False,
    style={"description_width": "initial"},
)
lambda_rep = widgets.FloatSlider(
    value=1.0, min=0.0, max=2.0, step=0.1,
    readout_format=".1f", description="Repulsion",
    continuous_update=False,
    style={"description_width": "initial"},
)
negative_ratio = widgets.IntSlider(
    value=4, min=0, max=10, step=1,
    description="Negative ratio", continuous_update=False,
    style={"description_width": "initial"},
)

run_button = widgets.Button(
    description="Fit embedding",
    button_style="primary",
    icon="play",
)
output = widgets.Output()


def fit_and_display(_=None):
    run_button.disabled = True
    with output:
        clear_output(wait=True)
        try:
            n_samples = sample_count.value
            X = X_pool[:n_samples]
            labels = labels_pool[:n_samples]
            estimator = IteratedMSTEmbedding(
                n_msts=n_msts.value,
                rank_weight_exponent=rank_exponent.value,
                n_components=2,
                n_epochs=n_epochs.value,
                learning_rate=learning_rate.value,
                negative_ratio=negative_ratio.value,
                lambda_rep=lambda_rep.value,
                random_state=42,
                device="auto",
            )
            started = time.perf_counter()
            embedding = estimator.fit_transform(X)
            elapsed = time.perf_counter() - started

            fig, ax = plt.subplots(figsize=(8, 6))
            points = ax.scatter(
                embedding[:, 0], embedding[:, 1],
                c=labels, cmap="tab10", vmin=-0.5, vmax=9.5,
                s=10, alpha=0.8, linewidths=0,
            )
            ax.set(
                title=(
                    f"2D MST embedding · {n_samples:,} samples · "
                    f"{n_msts.value} MSTs · α={rank_exponent.value:g} · "
                    f"{n_epochs.value} epochs"
                ),
                xlabel="Embedding dimension 1",
                ylabel="Embedding dimension 2",
            )
            fig.colorbar(points, ax=ax, ticks=range(10), label="Digit")
            fig.tight_layout()
            display(fig)
            plt.close(fig)
            display(HTML(
                f"<p>Fit in {elapsed:.1f} seconds on <b>{estimator.device_}</b>.</p>"
            ))
        except Exception:
            traceback.print_exc()
        finally:
            run_button.disabled = False


run_button.on_click(fit_and_display)
controls = widgets.VBox([
    widgets.HBox([sample_count, n_msts, rank_exponent]),
    widgets.HBox([n_epochs, learning_rate, lambda_rep, negative_ratio]),
])
display(controls, run_button, output)
